In [25]:
import os
import glob
import pandas as pd
import geopandas as gpd

file_inputs = [
    {"path": "data/alabama2026.geojson", "postal": "AL"},
    {"path": "data/tennessee2026.geojson", "postal": "TN"} 
]

# Standard target CRS (EPSG:4326 is standard for GeoJSON)
TARGET_CRS = "EPSG:4326"
all_gdfs = []

for state in file_inputs:
    gdf = gpd.read_file(state["path"])

    if gdf.crs != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)
        
    # Create matching key for the geojson/csv files with state postal and district number
    gdf["District"] = state["postal"] + gdf["id"].astype(str).str.zfill(2)
    
    gdf = gdf[["id", "District", "DemPct", "RepPct", "Margin","geometry"]]
    
    all_gdfs.append(gdf)

states_gdf = pd.concat(all_gdfs, ignore_index=True)
states_gdf.head(8)

,id,District,DemPct,RepPct,Margin,geometry
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ..."
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ..."
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ..."
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ..."
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ..."
5,6,AL06,0.320528,0.662526,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ..."
6,7,AL07,0.582883,0.405147,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ..."
7,1,TN01,0.205668,0.784080,-0.294332,"MULTIPOLYGON (((-83.46710 36.17469, -83.47240 ..."


In [26]:
# 2024 presidential results by Congressional district
pres24 = pd.read_csv("data/2024_pres.csv")

# Merged desired columns from the csv file into the geojson file
columns_to_keep = ['District', 'Harris', 'Trump 24', 'Total', 'Margin']

pres24 = pres24[columns_to_keep]

# Inner join the datasets on matching District values
merged_gdf = states_gdf.merge(pres24, on='District', how='inner')
merged_gdf.head(8)

,id,District,DemPct,RepPct,Margin_x,geometry,Harris,Trump 24,Total,Margin_y
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",73003,257060,332700,-55.32%
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",155603,131721,290033,8.23%
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",82654,229676,314869,-46.69%
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",53098,267953,323449,-66.43%
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",122344,224704,351650,-29.11%
5,6,AL06,0.320528,0.662526,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ...",105388,239984,349125,-38.55%
6,7,AL07,0.582883,0.405147,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ...",180321,111517,294526,23.36%
7,1,TN01,0.205668,0.784080,-0.294332,"MULTIPOLYGON (((-83.46710 36.17469, -83.47240 ...",71649,273159,348378,-57.84%


In [27]:
# rename columns
gdf_clean = merged_gdf.rename(columns={
    'id': 'District No.',
    'DemPct': 'Harris New',
    'RepPct': 'Trump New',
    'Margin_x': 'Partisan Index',
    'Harris': 'Harris Votes',
    'Trump 24': 'Trump Votes',
    'Margin_y': 'Margin %',
})

gdf_clean.head(8)

,District No.,District,Harris New,Trump New,Partisan Index,geometry,Harris Votes,Trump Votes,Total,Margin %
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",73003,257060,332700,-55.32%
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",155603,131721,290033,8.23%
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",82654,229676,314869,-46.69%
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",53098,267953,323449,-66.43%
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",122344,224704,351650,-29.11%
5,6,AL06,0.320528,0.662526,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ...",105388,239984,349125,-38.55%
6,7,AL07,0.582883,0.405147,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ...",180321,111517,294526,23.36%
7,1,TN01,0.205668,0.784080,-0.294332,"MULTIPOLYGON (((-83.46710 36.17469, -83.47240 ...",71649,273159,348378,-57.84%


In [28]:
# Create new columns converting the raw votes into percentages, as well as show the percent margin difference
gdf_clean['Harris 24'] = gdf_clean['Harris Votes'] / gdf_clean['Total']
gdf_clean['Trump 24'] = gdf_clean['Trump Votes'] / gdf_clean['Total']

gdf_clean['Margin 24'] = gdf_clean['Harris 24'] - gdf_clean['Trump 24']
gdf_clean['Margin New'] = gdf_clean['Harris New'] - gdf_clean['Trump New']

gdf_clean['Margin Shift'] = gdf_clean['Margin New'] - gdf_clean['Margin 24']
gdf_clean[['Margin New %', 'Margin Shift %']] = gdf_clean[['Margin New', 'Margin Shift']].map('{:.2%}'.format)

# Remove the columns that we no longer need
gdf_clean = gdf_clean.drop(columns=['Harris Votes', 'Trump Votes', 'Total'])

gdf_clean.head()

,District No.,District,Harris New,Trump New,Partisan Index,geometry,Margin %,Harris 24,Trump 24,Margin 24,Margin New,Margin Shift,Margin New %,Margin Shift %
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",-55.32%,0.219426,0.772648,-0.553222,-0.358934,0.194288,-35.89%,19.43%
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",8.23%,0.536501,0.454159,0.082342,-0.143004,-0.225346,-14.30%,-22.53%
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",-46.69%,0.262503,0.729434,-0.466931,-0.465175,0.001756,-46.52%,0.18%
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",-66.43%,0.164162,0.828424,-0.664262,-0.663093,0.001169,-66.31%,0.12%
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",-29.11%,0.347914,0.638999,-0.291085,-0.289816,0.001269,-28.98%,0.13%


In [30]:
# Column to indicate districts targeted by redistricting
gdf_clean['Targeted'] = (gdf_clean['Margin New'] * gdf_clean['Margin 24']) < 0

# Reorder the columns
reordered_columns = ['District No.', 'District', 'Harris New', 'Trump New', 'Margin New', 'Margin %',
                    'Margin Shift', 'Margin Shift %', 'Partisan Index',
                    'Harris 24', 'Trump 24', 'Margin 24', 'Margin %', 'Targeted', 'geometry']

gdf_clean = gdf_clean[reordered_columns]

gdf_clean.to_file("data/tennebama.geojson", driver='GeoJSON')
gdf_clean.head(16)

,District No.,District,Harris New,Trump New,Margin New,Margin %,Margin %,Margin Shift,Margin Shift %,Partisan Index,Harris 24,Trump 24,Margin 24,Margin %,Margin %,Targeted,geometry
0,1,AL01,0.314327,0.673261,-0.358934,-55.32%,-55.32%,0.194288,19.43%,-0.185673,0.219426,0.772648,-0.553222,-55.32%,-55.32%,False,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ..."
1,2,AL02,0.422827,0.565831,-0.143004,8.23%,8.23%,-0.225346,-22.53%,-0.077173,0.536501,0.454159,0.082342,8.23%,8.23%,True,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ..."
2,3,AL03,0.261882,0.727057,-0.465175,-46.69%,-46.69%,0.001756,0.18%,-0.238118,0.262503,0.729434,-0.466931,-46.69%,-46.69%,False,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ..."
3,4,AL04,0.163376,0.826469,-0.663093,-66.43%,-66.43%,0.001169,0.12%,-0.336624,0.164162,0.828424,-0.664262,-66.43%,-66.43%,False,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ..."
4,5,AL05,0.346103,0.635919,-0.289816,-29.11%,-29.11%,0.001269,0.13%,-0.153897,0.347914,0.638999,-0.291085,-29.11%,-29.11%,False,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ..."
5,6,AL06,0.320528,0.662526,-0.341998,-38.55%,-38.55%,0.043526,4.35%,-0.179472,0.301863,0.687387,-0.385524,-38.55%,-38.55%,False,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ..."
6,7,AL07,0.582883,0.405147,0.177735,23.36%,23.36%,-0.055874,-5.59%,0.082883,0.612241,0.378632,0.233609,23.36%,23.36%,False,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ..."
7,1,TN01,0.205668,0.784080,-0.578412,-57.84%,-57.84%,0.000011,0.00%,-0.294332,0.205665,0.784088,-0.578423,-57.84%,-57.84%,False,"MULTIPOLYGON (((-83.46710 36.17469, -83.47240 ..."
8,2,TN02,0.323837,0.662405,-0.338568,-33.86%,-33.86%,-0.000009,-0.00%,-0.176163,0.323839,0.662398,-0.338559,-33.86%,-33.86%,False,"MULTIPOLYGON (((-84.32691 35.66496, -84.32648 ..."
9,3,TN03,0.314765,0.672058,-0.357294,-35.88%,-35.88%,0.001462,0.15%,-0.185235,0.314071,0.672827,-0.358755,-35.88%,-35.88%,False,"MULTIPOLYGON (((-85.18892 35.41800, -85.19288 ..."
